# DreamerV3 World Model — Full GPU Training
Upload this to Colab with GPU runtime. It sets up the full project and trains on Pendulum with CUDA-accelerated matmul.

In [ ]:
!nvidia-smi
!pip install gymnasium -q

In [ ]:
import os
os.makedirs('nn', exist_ok=True)
os.makedirs('model', exist_ok=True)
os.makedirs('agent', exist_ok=True)
os.makedirs('training', exist_ok=True)
os.makedirs('gpu_ops', exist_ok=True)
for d in ['nn', 'model', 'agent', 'training', 'gpu_ops']:
    open(f'{d}/__init__.py', 'w').close()
print('Directories created')

In [ ]:
%%writefile nn/tensor.py
import numpy as np
import sys, os
sys.path.insert(0, os.path.join(os.path.dirname(os.path.abspath(__file__)), '..'))
try:
    from gpu_ops.backend import matmul as _backend_matmul
except (ImportError, ModuleNotFoundError):
    _backend_matmul = np.matmul


def _unbroadcast(grad, shape):
    while grad.ndim > len(shape):
        grad = grad.sum(axis=0)
    for i, (g, s) in enumerate(zip(grad.shape, shape)):
        if s == 1 and g != 1:
            grad = grad.sum(axis=i, keepdims=True)
    return grad


class Tensor:
    def __init__(self, data, requires_grad=False, _children=(), _op=''):
        if isinstance(data, np.ndarray):
            self.data = data if data.dtype == np.float32 else data.astype(np.float32)
        else:
            self.data = np.array(data, dtype=np.float32)
        self.requires_grad = requires_grad
        self.grad = None
        self._backward = lambda: None
        self._children = set(_children)
        self._op = _op

    @property
    def shape(self):
        return self.data.shape

    def zero_grad(self):
        self.grad = None

    def detach(self):
        return Tensor(self.data.copy(), requires_grad=False)

    def backward(self):
        topo = []
        visited = set()
        def build(node):
            if node not in visited:
                visited.add(node)
                for child in node._children:
                    build(child)
                topo.append(node)
        build(self)
        self.grad = np.ones_like(self.data)
        for node in reversed(topo):
            node._backward()

    def __add__(self, other):
        other = other if isinstance(other, Tensor) else Tensor(other)
        out = Tensor(self.data + other.data, requires_grad=self.requires_grad or other.requires_grad, _children=(self, other), _op='+')
        def _backward():
            if self.requires_grad:
                g = _unbroadcast(out.grad, self.shape)
                self.grad = g if self.grad is None else self.grad + g
            if other.requires_grad:
                g = _unbroadcast(out.grad, other.shape)
                other.grad = g if other.grad is None else other.grad + g
        out._backward = _backward
        return out

    def __radd__(self, other):
        return self.__add__(other)

    def __sub__(self, other):
        other = other if isinstance(other, Tensor) else Tensor(other)
        out = Tensor(self.data - other.data, requires_grad=self.requires_grad or other.requires_grad, _children=(self, other), _op='-')
        def _backward():
            if self.requires_grad:
                g = _unbroadcast(out.grad, self.shape)
                self.grad = g if self.grad is None else self.grad + g
            if other.requires_grad:
                g = _unbroadcast(-out.grad, other.shape)
                other.grad = g if other.grad is None else other.grad + g
        out._backward = _backward
        return out

    def __rsub__(self, other):
        other = other if isinstance(other, Tensor) else Tensor(other)
        return other.__sub__(self)

    def __mul__(self, other):
        other = other if isinstance(other, Tensor) else Tensor(other)
        out = Tensor(self.data * other.data, requires_grad=self.requires_grad or other.requires_grad, _children=(self, other), _op='*')
        def _backward():
            if self.requires_grad:
                g = _unbroadcast(out.grad * other.data, self.shape)
                self.grad = g if self.grad is None else self.grad + g
            if other.requires_grad:
                g = _unbroadcast(out.grad * self.data, other.shape)
                other.grad = g if other.grad is None else other.grad + g
        out._backward = _backward
        return out

    def __rmul__(self, other):
        return self.__mul__(other)

    def __neg__(self):
        return self * (-1)

    def __pow__(self, power):
        out = Tensor(self.data ** power, requires_grad=self.requires_grad, _children=(self,), _op=f'**{power}')
        def _backward():
            if self.requires_grad:
                g = out.grad * power * self.data ** (power - 1)
                self.grad = g if self.grad is None else self.grad + g
        out._backward = _backward
        return out

    def matmul(self, other):
        other = other if isinstance(other, Tensor) else Tensor(other)
        out = Tensor(
            _backend_matmul(self.data, other.data),
            requires_grad=self.requires_grad or other.requires_grad,
            _children=(self, other), _op='matmul',
        )
        def _backward():
            if self.requires_grad:
                g = out.grad @ np.swapaxes(other.data, -1, -2)
                self.grad = g if self.grad is None else self.grad + g
            if other.requires_grad:
                g = np.swapaxes(self.data, -1, -2) @ out.grad
                other.grad = g if other.grad is None else other.grad + g
        out._backward = _backward
        return out

    def sum(self, axis=None, keepdims=False):
        out = Tensor(self.data.sum(axis=axis, keepdims=keepdims), requires_grad=self.requires_grad, _children=(self,), _op='sum')
        def _backward():
            if self.requires_grad:
                g = np.ones_like(self.data) * out.grad if axis is None else np.expand_dims(out.grad, axis=axis) * np.ones_like(self.data) if not keepdims else out.grad * np.ones_like(self.data)
                self.grad = g if self.grad is None else self.grad + g
        out._backward = _backward
        return out

    def mean(self, axis=None, keepdims=False):
        n = self.data.size if axis is None else self.data.shape[axis]
        return self.sum(axis=axis, keepdims=keepdims) * (1.0 / n)

    def reshape(self, *shape):
        if len(shape) == 1 and isinstance(shape[0], tuple):
            shape = shape[0]
        out = Tensor(self.data.reshape(shape), requires_grad=self.requires_grad, _children=(self,), _op='reshape')
        def _backward():
            if self.requires_grad:
                g = out.grad.reshape(self.shape)
                self.grad = g if self.grad is None else self.grad + g
        out._backward = _backward
        return out

    def sigmoid(self):
        s = 1.0 / (1.0 + np.exp(-self.data))
        out = Tensor(s, requires_grad=self.requires_grad, _children=(self,), _op='sigmoid')
        def _backward():
            if self.requires_grad:
                g = out.grad * s * (1 - s)
                self.grad = g if self.grad is None else self.grad + g
        out._backward = _backward
        return out

    def tanh(self):
        t = np.tanh(self.data)
        out = Tensor(t, requires_grad=self.requires_grad, _children=(self,), _op='tanh')
        def _backward():
            if self.requires_grad:
                g = out.grad * (1 - t ** 2)
                self.grad = g if self.grad is None else self.grad + g
        out._backward = _backward
        return out

    def relu(self):
        out = Tensor(np.maximum(0, self.data), requires_grad=self.requires_grad, _children=(self,), _op='relu')
        def _backward():
            if self.requires_grad:
                g = out.grad * (self.data > 0).astype(np.float32)
                self.grad = g if self.grad is None else self.grad + g
        out._backward = _backward
        return out

    def silu(self):
        s = 1.0 / (1.0 + np.exp(-self.data))
        out = Tensor(self.data * s, requires_grad=self.requires_grad, _children=(self,), _op='silu')
        def _backward():
            if self.requires_grad:
                g = out.grad * (s + self.data * s * (1 - s))
                self.grad = g if self.grad is None else self.grad + g
        out._backward = _backward
        return out

    def softmax(self, axis=-1):
        e = np.exp(self.data - self.data.max(axis=axis, keepdims=True))
        s = e / e.sum(axis=axis, keepdims=True)
        out = Tensor(s, requires_grad=self.requires_grad, _children=(self,), _op='softmax')
        def _backward():
            if self.requires_grad:
                g = out.grad * s - s * (out.grad * s).sum(axis=axis, keepdims=True)
                self.grad = g if self.grad is None else self.grad + g
        out._backward = _backward
        return out

    def log(self):
        out = Tensor(np.log(self.data + 1e-8), requires_grad=self.requires_grad, _children=(self,), _op='log')
        def _backward():
            if self.requires_grad:
                g = out.grad / (self.data + 1e-8)
                self.grad = g if self.grad is None else self.grad + g
        out._backward = _backward
        return out

    def clamp(self, min_val=None, max_val=None):
        data = self.data.copy()
        if min_val is not None:
            data = np.maximum(data, min_val)
        if max_val is not None:
            data = np.minimum(data, max_val)
        out = Tensor(data, requires_grad=self.requires_grad, _children=(self,), _op='clamp')
        def _backward():
            if self.requires_grad:
                mask = np.ones_like(self.data)
                if min_val is not None:
                    mask *= (self.data >= min_val)
                if max_val is not None:
                    mask *= (self.data <= max_val)
                g = out.grad * mask
                self.grad = g if self.grad is None else self.grad + g
        out._backward = _backward
        return out

    def symlog(self):
        out = Tensor(np.sign(self.data) * np.log(1 + np.abs(self.data)), requires_grad=self.requires_grad, _children=(self,), _op='symlog')
        def _backward():
            if self.requires_grad:
                g = out.grad / (1 + np.abs(self.data))
                self.grad = g if self.grad is None else self.grad + g
        out._backward = _backward
        return out

    def straight_through(self, hard):
        out = Tensor(hard, requires_grad=self.requires_grad, _children=(self,), _op='st')
        def _backward():
            if self.requires_grad:
                self.grad = out.grad.copy() if self.grad is None else self.grad + out.grad
        out._backward = _backward
        return out


def cat(tensors, axis=0):
    data = np.concatenate([t.data for t in tensors], axis=axis)
    out = Tensor(data, requires_grad=any(t.requires_grad for t in tensors), _children=tuple(tensors), _op='cat')
    def _backward():
        if out.grad is None:
            return
        sections = np.cumsum([t.shape[axis] for t in tensors[:-1]])
        grads = np.split(out.grad, sections, axis=axis)
        for t, g in zip(tensors, grads):
            if t.requires_grad:
                t.grad = g.copy() if t.grad is None else t.grad + g
    out._backward = _backward
    return out


def one_hot(indices, num_classes):
    flat = indices.reshape(-1)
    oh = np.zeros((flat.size, num_classes), dtype=np.float32)
    oh[np.arange(flat.size), flat] = 1.0
    return oh.reshape(indices.shape + (num_classes,))

In [ ]:
%%writefile nn/linear.py
import numpy as np
from nn.tensor import Tensor

class Linear:
    def __init__(self, in_features, out_features, bias=True):
        scale = (2.0 / in_features) ** 0.5
        self.weight = Tensor(np.random.randn(in_features, out_features).astype(np.float32) * scale, requires_grad=True)
        self.bias = None
        if bias:
            self.bias = Tensor(np.zeros(out_features, dtype=np.float32), requires_grad=True)

    def __call__(self, x):
        out = x.matmul(self.weight)
        if self.bias is not None:
            out = out + self.bias
        return out

    def parameters(self):
        params = [self.weight]
        if self.bias is not None:
            params.append(self.bias)
        return params

In [ ]:
%%writefile nn/mlp.py
from nn.linear import Linear

class MLP:
    def __init__(self, sizes, activation='silu'):
        self.activation = activation
        self.layers = [Linear(sizes[i], sizes[i+1]) for i in range(len(sizes)-1)]

    def __call__(self, x):
        for i, layer in enumerate(self.layers):
            x = layer(x)
            if i < len(self.layers) - 1:
                x = x.silu() if self.activation == 'silu' else x.relu()
        return x

    def parameters(self):
        params = []
        for layer in self.layers:
            params.extend(layer.parameters())
        return params

In [ ]:
%%writefile nn/optimizer.py
import numpy as np

class AdamW:
    def __init__(self, parameters, lr=3e-4, betas=(0.9, 0.999), eps=1e-8, weight_decay=0.01, grad_clip=100.0):
        self.parameters = parameters
        self.lr = lr
        self.beta1, self.beta2 = betas
        self.eps = eps
        self.weight_decay = weight_decay
        self.grad_clip = grad_clip
        self.t = 0
        self.m = [np.zeros_like(p.data) for p in self.parameters]
        self.v = [np.zeros_like(p.data) for p in self.parameters]

    def step(self):
        self.t += 1
        if self.grad_clip > 0:
            total_norm = sum(np.sum(p.grad ** 2) for p in self.parameters if p.grad is not None)
            total_norm = np.sqrt(total_norm)
            clip_coef = self.grad_clip / (total_norm + 1e-6)
            if clip_coef < 1.0:
                for p in self.parameters:
                    if p.grad is not None:
                        p.grad = p.grad * clip_coef
        for i, p in enumerate(self.parameters):
            if p.grad is None:
                continue
            self.m[i] = self.beta1 * self.m[i] + (1 - self.beta1) * p.grad
            self.v[i] = self.beta2 * self.v[i] + (1 - self.beta2) * (p.grad ** 2)
            m_hat = self.m[i] / (1 - self.beta1 ** self.t)
            v_hat = self.v[i] / (1 - self.beta2 ** self.t)
            p.data -= self.lr * self.weight_decay * p.data
            p.data -= self.lr * m_hat / (np.sqrt(v_hat) + self.eps)

    def zero_grad(self):
        for p in self.parameters:
            p.zero_grad()

In [ ]:
%%writefile nn/gru_cell.py
import numpy as np
from nn.linear import Linear
from nn.tensor import Tensor, cat

class GRUCell:
    def __init__(self, input_size, hidden_size):
        self.hidden_size = hidden_size
        self.reset_gate = Linear(input_size + hidden_size, hidden_size)
        self.update_gate = Linear(input_size + hidden_size, hidden_size)
        self.candidate = Linear(input_size + hidden_size, hidden_size)

    def __call__(self, x, h):
        if h is None:
            h = Tensor(np.zeros((x.shape[0], self.hidden_size), dtype=np.float32))
        combined = cat([x, h], axis=1)
        r = self.reset_gate(combined).sigmoid()
        z = self.update_gate(combined).sigmoid()
        h_reset = r * h
        combined_r = cat([x, h_reset], axis=1)
        candidate = self.candidate(combined_r).tanh()
        h_new = z * h + (1 - z) * candidate
        return h_new

    def parameters(self):
        params = []
        for layer in [self.reset_gate, self.update_gate, self.candidate]:
            params.extend(layer.parameters())
        return params

In [ ]:
%%writefile model/encoder.py
from nn.mlp import MLP

class Encoder:
    def __init__(self, obs_size, hidden_size=400, latent_size=200):
        self.net = MLP([obs_size, hidden_size, hidden_size, latent_size])
    def __call__(self, obs):
        return self.net(obs)
    def parameters(self):
        return self.net.parameters()

In [ ]:
%%writefile model/rssm.py
import numpy as np
from nn.tensor import Tensor, cat, one_hot
from nn.linear import Linear
from nn.mlp import MLP
from nn.gru_cell import GRUCell

class RSSM:
    def __init__(self, obs_size, action_size, hidden_size=200, stoch_size=32, stoch_classes=32):
        self.hidden_size = hidden_size
        self.stoch_size = stoch_size
        self.stoch_classes = stoch_classes
        stoch_flat = stoch_size * stoch_classes
        self.pre_gru = Linear(stoch_flat + action_size, hidden_size)
        self.gru = GRUCell(hidden_size, hidden_size)
        self.prior = MLP([hidden_size, hidden_size, stoch_flat])
        self.posterior = MLP([hidden_size + obs_size, hidden_size, stoch_flat])

    def forward(self, prev_h, prev_z, action, encoded_obs=None):
        if prev_z is None:
            prev_z = Tensor(np.zeros((action.shape[0], self.stoch_size * self.stoch_classes), dtype=np.float32))
        if prev_h is None:
            prev_h = Tensor(np.zeros((action.shape[0], self.hidden_size), dtype=np.float32))
        gru_input = self.pre_gru(cat([prev_z, action], axis=1)).silu()
        h = self.gru(gru_input, prev_h)
        prior_logits = self.prior(h).reshape(-1, self.stoch_size, self.stoch_classes)
        if encoded_obs is not None:
            post_logits = self.posterior(cat([h, encoded_obs], axis=1)).reshape(-1, self.stoch_size, self.stoch_classes)
            logits = post_logits
        else:
            post_logits = None
            logits = prior_logits
        probs = logits.softmax(axis=-1)
        indices = np.array([[np.random.choice(self.stoch_classes, p=probs.data[b, d]) for d in range(self.stoch_size)] for b in range(probs.shape[0])])
        hard = one_hot(indices, self.stoch_classes)
        z = probs.straight_through(hard)
        z_flat = z.reshape(-1, self.stoch_size * self.stoch_classes)
        return h, z_flat, prior_logits, post_logits

    def parameters(self):
        params = self.pre_gru.parameters()
        params.extend(self.gru.parameters())
        params.extend(self.prior.parameters())
        params.extend(self.posterior.parameters())
        return params

In [ ]:
%%writefile model/decoder.py
from nn.mlp import MLP
from nn.tensor import cat

class Decoder:
    def __init__(self, state_size, obs_size, hidden_size=400):
        self.net = MLP([state_size, hidden_size, hidden_size, obs_size])
    def __call__(self, h, z):
        return self.net(cat([h, z], axis=1))
    def parameters(self):
        return self.net.parameters()

In [ ]:
%%writefile model/reward_model.py
from nn.mlp import MLP
from nn.tensor import cat

class RewardModel:
    def __init__(self, state_size, hidden_size=400):
        self.net = MLP([state_size, hidden_size, hidden_size, 1])
    def __call__(self, h, z):
        return self.net(cat([h, z], axis=1))
    def parameters(self):
        return self.net.parameters()

In [ ]:
%%writefile model/continue_model.py
from nn.mlp import MLP
from nn.tensor import cat

class ContinueModel:
    def __init__(self, state_size, hidden_size=400):
        self.net = MLP([state_size, hidden_size, hidden_size, 1])
    def __call__(self, h, z):
        return self.net(cat([h, z], axis=1)).sigmoid()
    def parameters(self):
        return self.net.parameters()

In [ ]:
%%writefile model/world_model.py
from nn.tensor import Tensor, cat
from model.encoder import Encoder
from model.rssm import RSSM
from model.decoder import Decoder
from model.reward_model import RewardModel
from model.continue_model import ContinueModel

class WorldModel:
    def __init__(self, obs_size, action_size, hidden_size=200, stoch_size=32, stoch_classes=32):
        state_size = hidden_size + stoch_size * stoch_classes
        self.encoder = Encoder(obs_size, hidden_size, hidden_size)
        self.rssm = RSSM(hidden_size, action_size, hidden_size, stoch_size, stoch_classes)
        self.decoder = Decoder(state_size, obs_size, hidden_size)
        self.reward_model = RewardModel(state_size, hidden_size)
        self.continue_model = ContinueModel(state_size, hidden_size)

    def forward(self, observations, actions, rewards, continues):
        seq_len = len(observations)
        h, z = None, None
        all_h, all_z, all_prior, all_post = [], [], [], []
        for t in range(seq_len):
            encoded = self.encoder(observations[t])
            h, z, prior_logits, post_logits = self.rssm.forward(h, z, actions[t], encoded_obs=encoded)
            all_h.append(h); all_z.append(z)
            all_prior.append(prior_logits); all_post.append(post_logits)
        H = cat(all_h, axis=0)
        Z = cat(all_z, axis=0)
        recon = self.decoder(H, Z)
        obs_target = cat(observations, axis=0)
        recon_loss = ((recon - obs_target.symlog()) ** 2).mean()
        pred_reward = self.reward_model(H, Z)
        reward_target = cat(rewards, axis=0)
        reward_loss = ((pred_reward - reward_target.symlog()) ** 2).mean()
        pred_continue = self.continue_model(H, Z)
        continue_target = cat(continues, axis=0)
        continue_loss = ((pred_continue - continue_target) ** 2).mean()
        prior = cat(all_prior, axis=0)
        post = cat(all_post, axis=0)
        prior_probs = prior.softmax(axis=-1)
        post_probs = post.softmax(axis=-1)
        kl = (post_probs * (post_probs.log() - prior_probs.log())).sum(axis=-1).mean()
        kl_loss = kl.clamp(min_val=1.0)
        loss = recon_loss + reward_loss + continue_loss + kl_loss
        return loss

    def parameters(self):
        p = self.encoder.parameters()
        p.extend(self.rssm.parameters())
        p.extend(self.decoder.parameters())
        p.extend(self.reward_model.parameters())
        p.extend(self.continue_model.parameters())
        return p

In [ ]:
%%writefile agent/actor.py
from nn.mlp import MLP
from nn.tensor import cat

class Actor:
    def __init__(self, state_size, action_size, hidden_size=400):
        self.action_size = action_size
        self.net = MLP([state_size, hidden_size, hidden_size, action_size])
    def __call__(self, h, z):
        return self.net(cat([h, z], axis=1)).tanh()
    def parameters(self):
        return self.net.parameters()

In [ ]:
%%writefile agent/critic.py
from nn.mlp import MLP
from nn.tensor import cat

class Critic:
    def __init__(self, state_size, hidden_size=400):
        self.net = MLP([state_size, hidden_size, hidden_size, 1])
    def __call__(self, h, z):
        return self.net(cat([h, z], axis=1))
    def parameters(self):
        return self.net.parameters()

In [ ]:
%%writefile training/replay_buffer.py
import numpy as np

class ReplayBuffer:
    def __init__(self, capacity=100000):
        self.capacity = capacity
        self.observations = []
        self.actions = []
        self.rewards = []
        self.continues = []

    def add(self, obs, action, reward, done):
        self.observations.append(obs)
        self.actions.append(action)
        self.rewards.append(reward)
        self.continues.append(0.0 if done else 1.0)
        if len(self.observations) > self.capacity:
            self.observations.pop(0)
            self.actions.pop(0)
            self.rewards.pop(0)
            self.continues.pop(0)

    def sample(self, batch_size, seq_len):
        max_start = len(self.observations) - seq_len
        if max_start < 1:
            return None
        indices = np.random.randint(0, max_start, size=batch_size)
        obs_seqs, act_seqs, rew_seqs, cont_seqs = [], [], [], []
        for t in range(seq_len):
            obs_seqs.append(np.stack([self.observations[i+t] for i in indices]).astype(np.float32))
            act_seqs.append(np.stack([self.actions[i+t] for i in indices]).astype(np.float32))
            rew_seqs.append(np.stack([[self.rewards[i+t]] for i in indices]).astype(np.float32))
            cont_seqs.append(np.stack([[self.continues[i+t]] for i in indices]).astype(np.float32))
        return obs_seqs, act_seqs, rew_seqs, cont_seqs

    def __len__(self):
        return len(self.observations)

In [ ]:
%%writefile gpu_ops/backend.py
import numpy as np
import ctypes
import os

_matmul_func = None

def _try_load():
    global _matmul_func
    lib_dir = os.path.dirname(os.path.abspath(__file__))
    for name in ['matmul_tiled.so', 'matmul.so']:
        path = os.path.join(lib_dir, name)
        if os.path.exists(path):
            try:
                lib = ctypes.CDLL(path)
                fn = 'cuda_matmul_tiled' if 'tiled' in name else 'cuda_matmul'
                _matmul_func = getattr(lib, fn)
                _matmul_func.argtypes = [ctypes.POINTER(ctypes.c_float)]*3 + [ctypes.c_int]*3
                _matmul_func.restype = None
                print(f'[backend] GPU matmul loaded: {name}')
                return
            except: pass

_try_load()

def matmul(a, b):
    if _matmul_func is not None and a.ndim == 2 and b.ndim == 2:
        a = np.ascontiguousarray(a, dtype=np.float32)
        b = np.ascontiguousarray(b, dtype=np.float32)
        M, K = a.shape
        N = b.shape[1]
        c = np.empty((M, N), dtype=np.float32)
        _matmul_func(
            a.ctypes.data_as(ctypes.POINTER(ctypes.c_float)),
            b.ctypes.data_as(ctypes.POINTER(ctypes.c_float)),
            c.ctypes.data_as(ctypes.POINTER(ctypes.c_float)),
            M, K, N)
        return c
    return np.matmul(a, b)

In [ ]:
%%writefile gpu_ops/matmul_tiled.cu
#include <cuda_runtime.h>
#define TILE 16
__global__ void matmul_tiled_kernel(float *A, float *B, float *C, int M, int K, int N) {
    __shared__ float tile_A[TILE][TILE];
    __shared__ float tile_B[TILE][TILE];
    int row = blockIdx.y * TILE + threadIdx.y;
    int col = blockIdx.x * TILE + threadIdx.x;
    float sum = 0.0f;
    for (int t = 0; t < (K + TILE - 1) / TILE; t++) {
        int a_col = t * TILE + threadIdx.x;
        tile_A[threadIdx.y][threadIdx.x] = (row < M && a_col < K) ? A[row*K+a_col] : 0.0f;
        int b_row = t * TILE + threadIdx.y;
        tile_B[threadIdx.y][threadIdx.x] = (b_row < K && col < N) ? B[b_row*N+col] : 0.0f;
        __syncthreads();
        for (int k = 0; k < TILE; k++) sum += tile_A[threadIdx.y][k] * tile_B[k][threadIdx.x];
        __syncthreads();
    }
    if (row < M && col < N) C[row*N+col] = sum;
}
extern "C" {
void cuda_matmul_tiled(float *A, float *B, float *C, int M, int K, int N) {
    float *d_A, *d_B, *d_C;
    cudaMalloc(&d_A, M*K*4); cudaMalloc(&d_B, K*N*4); cudaMalloc(&d_C, M*N*4);
    cudaMemcpy(d_A, A, M*K*4, cudaMemcpyHostToDevice);
    cudaMemcpy(d_B, B, K*N*4, cudaMemcpyHostToDevice);
    dim3 t(TILE,TILE), b((N+TILE-1)/TILE,(M+TILE-1)/TILE);
    matmul_tiled_kernel<<<b,t>>>(d_A,d_B,d_C,M,K,N);
    cudaMemcpy(C, d_C, M*N*4, cudaMemcpyDeviceToHost);
    cudaFree(d_A); cudaFree(d_B); cudaFree(d_C);
}
}

In [ ]:
!nvcc -shared -Xcompiler -fPIC -o gpu_ops/matmul_tiled.so gpu_ops/matmul_tiled.cu -Wno-deprecated-gpu-targets
print('CUDA kernel compiled!')

In [ ]:
import sys, time
sys.path.insert(0, '/content')

import numpy as np
import gymnasium as gym
from nn.tensor import Tensor, cat
from nn.optimizer import AdamW
from model.world_model import WorldModel
from agent.actor import Actor
from agent.critic import Critic
from training.replay_buffer import ReplayBuffer

print('All imports OK')

In [ ]:
env = gym.make('Pendulum-v1')
obs_size = env.observation_space.shape[0]
action_size = env.action_space.shape[0]

hidden_size = 128
stoch_size = 16
stoch_classes = 16
state_size = hidden_size + stoch_size * stoch_classes

world_model = WorldModel(obs_size, action_size, hidden_size, stoch_size, stoch_classes)
actor = Actor(state_size, action_size, hidden_size=128)
critic = Critic(state_size, hidden_size=128)

wm_opt = AdamW(world_model.parameters(), lr=3e-4, grad_clip=100.0)
actor_opt = AdamW(actor.parameters(), lr=1e-4, grad_clip=100.0)
critic_opt = AdamW(critic.parameters(), lr=1e-4, grad_clip=100.0)
buffer = ReplayBuffer(capacity=50000)

print(f'Model params: {sum(p.data.size for p in world_model.parameters()):,}')
print(f'Actor params: {sum(p.data.size for p in actor.parameters()):,}')
print(f'Critic params: {sum(p.data.size for p in critic.parameters()):,}')
print('Setup complete!')

In [ ]:
def collect_data(env, buffer, world_model=None, actor=None, num_steps=1000):
    obs, _ = env.reset()
    h, z = None, None
    for _ in range(num_steps):
        if actor is not None and world_model is not None:
            obs_tensor = Tensor(obs.reshape(1, -1).astype(np.float32))
            encoded = world_model.encoder(obs_tensor)
            prev_act = Tensor(np.zeros((1, env.action_space.shape[0]), dtype=np.float32))
            h, z, _, _ = world_model.rssm.forward(h, z, prev_act, encoded_obs=encoded)
            action = actor(h.detach(), z.detach())
            act_np = np.clip(action.data[0], -1, 1) * 2.0
            noise = np.random.randn(*act_np.shape).astype(np.float32) * 0.3
            act_np = np.clip(act_np + noise, -2, 2)
        else:
            act_np = env.action_space.sample()
        next_obs, reward, terminated, truncated, _ = env.step(act_np)
        buffer.add(obs, act_np, reward, terminated or truncated)
        obs = next_obs
        if terminated or truncated:
            obs, _ = env.reset()
            h, z = None, None

def train_world_model(world_model, optimizer, buffer, batch_size=16, seq_len=16):
    batch = buffer.sample(batch_size, seq_len)
    if batch is None: return None
    obs_seqs, act_seqs, rew_seqs, cont_seqs = batch
    observations = [Tensor(o) for o in obs_seqs]
    actions = [Tensor(a) for a in act_seqs]
    rewards = [Tensor(r) for r in rew_seqs]
    continues = [Tensor(c) for c in cont_seqs]
    optimizer.zero_grad()
    loss = world_model.forward(observations, actions, rewards, continues)
    loss.backward()
    optimizer.step()
    return float(loss.data)

def train_actor_critic(world_model, actor, critic, actor_opt, critic_opt, buffer, horizon=15, batch_size=16):
    batch = buffer.sample(batch_size, 1)
    if batch is None: return
    obs_start = Tensor(batch[0][0])
    act_start = Tensor(batch[1][0])
    encoded = world_model.encoder(obs_start)
    h, z, _, _ = world_model.rssm.forward(None, None, act_start, encoded_obs=encoded)
    h, z = h.detach(), z.detach()
    imagined_h, imagined_z = [h], [z]
    rewards, continues = [], []
    for _ in range(horizon):
        action = actor(h, z)
        h, z, _, _ = world_model.rssm.forward(h, z, action, encoded_obs=None)
        rewards.append(world_model.reward_model(h, z))
        continues.append(world_model.continue_model(h, z))
        imagined_h.append(h); imagined_z.append(z)
    values = [critic(h, z) for h, z in zip(imagined_h, imagined_z)]
    returns = [values[-1]]
    for t in reversed(range(horizon)):
        ret = rewards[t] + continues[t] * 0.99 * (0.05 * values[t+1] + 0.95 * returns[0])
        returns.insert(0, ret)
    returns = returns[:-1]
    actor_loss = Tensor(0.0)
    critic_loss = Tensor(0.0)
    for t in range(horizon):
        actor_loss = actor_loss + (-returns[t]).mean()
        critic_loss = critic_loss + ((values[t] - returns[t].detach()) ** 2).mean()
    actor_opt.zero_grad(); actor_loss.backward(); actor_opt.step()
    critic_opt.zero_grad(); critic_loss.backward(); critic_opt.step()

def evaluate(env, world_model, actor, num_episodes=5):
    total = []
    for _ in range(num_episodes):
        obs, _ = env.reset()
        h, z = None, None
        ep_reward = 0
        for step in range(200):
            obs_t = Tensor(obs.reshape(1,-1).astype(np.float32))
            encoded = world_model.encoder(obs_t)
            prev_a = Tensor(np.zeros((1, env.action_space.shape[0]), dtype=np.float32))
            h, z, _, _ = world_model.rssm.forward(h, z, prev_a, encoded_obs=encoded)
            action = actor(h.detach(), z.detach())
            act_np = np.clip(action.data[0], -1, 1) * 2.0
            obs, reward, term, trunc, _ = env.step(act_np)
            ep_reward += reward
            if term or trunc: break
        total.append(ep_reward)
    return sum(total) / len(total)

print('Functions defined!')

In [ ]:
print('Collecting initial random data...')
collect_data(env, buffer, num_steps=2000)
print(f'Buffer: {len(buffer)}')

random_reward = evaluate(env, world_model, actor, num_episodes=5)
print(f'Before training: {random_reward:.1f}')
print()

num_epochs = 30
train_steps_per_epoch = 500
collect_steps_per_epoch = 300

start_time = time.time()
for epoch in range(num_epochs):
    epoch_start = time.time()
    wm_losses = []
    for step in range(train_steps_per_epoch):
        wm_loss = train_world_model(world_model, wm_opt, buffer, batch_size=16, seq_len=16)
        if wm_loss is not None:
            wm_losses.append(wm_loss)
        train_actor_critic(world_model, actor, critic, actor_opt, critic_opt, buffer, horizon=15, batch_size=16)

    avg_wm = sum(wm_losses) / len(wm_losses) if wm_losses else 0
    avg_reward = evaluate(env, world_model, actor, num_episodes=5)
    epoch_time = time.time() - epoch_start
    total_time = time.time() - start_time
    print(f'epoch {epoch:2d}: wm_loss={avg_wm:.4f}, reward={avg_reward:.1f}, '
          f'buffer={len(buffer)}, time={epoch_time:.1f}s (total {total_time:.0f}s)')

    collect_data(env, buffer, world_model, actor, num_steps=collect_steps_per_epoch)

final = evaluate(env, world_model, actor, num_episodes=10)
print(f'\nFinal avg reward = {final:.1f}')
print(f'Total time: {time.time() - start_time:.1f}s')